Here we will be implementing the ACPC (Affine Cyclically Permutable Codes) coding technique 

Concepts:

- Galois fields, Linear codes, Linear cosets, Characteristic for a field, Base field, Cyclic order for a codeword

- Lagrange's theorem for cyclic order (v|n), Block code length(n), Mersenne Prime

- KT construction to find a representative from each Cyclic equivalence class

- Cyclic codes, Cyclically permutable codes (CPC) or Cyclic Register Codes (CRC), Cyclic equivalence class

- BCH coding, minimal polynomial, irreducible polynomial, Cyclotomic coset, Frobenius Map (gives conjugate in GF)

- Affine transform

- Generation matrix(G), Parity-check matrix(H)

- Cyclic shift, How BCH codes makes use of it, error correction using BCH codes

- How the generator polynomial is selected and Generator matrix

- Systematic matrix

In [2]:
import numpy as np
import galois
import itertools
import random

- Defining the Galois Field we are going to use

- Mersenne number = 2**m-1

- var m alos defines the lowest degree polynomial of GF(2**m) over GF(2)

- Fixing the mersenne prime(n) so each of the cyclic equivalence class have possible cyclic order {1, n}

- we will be correcting upto (t=) 2 errors so we define the BCH code accordingly

- we will be defining number of information bitsfor BCH code later

- In GF2, -1 is equivalent to 1

# NOTE: 
- The actual number of information bits(B) are different from BCH code information bits (k_out)

In [24]:
m = 5
n = 2**m -1
print(f"Length of the block code: {n}")

t = 2
d_min = 2 * t + 1

GF32 = galois.GF(2**m)
print(GF32.properties)

# we are defining a varible in GF(2)
x = galois.Poly.Identity(galois.GF2)

one = galois.GF2(1)
x_n = x**n - one
print(f'The polynomial used for BCH code construction upto {t} error: {x_n}')

Length of the block code: 31
Galois Field:
  name: GF(2^5)
  characteristic: 2
  degree: 5
  order: 32
  irreducible_poly: x^5 + x^2 + 1
  is_primitive_poly: True
  primitive_element: x
The polynomial used for BCH code construction upto 2 error: x^31 + 1


- Identification of irreducible polynomials(factors) of Base Polynomial over which BCH code is defined

- Construction of Cyclotomic Cosets

- Selection of consecutive powers upto (2*t) of primitive elements to correct upto t errors

In [20]:
def con_coset(n):

    coset = {}
    leader = -1
    visited = set()

    for i in range(n):
        if i in visited:
            continue
        else:
            leader = i
            cos_lis = np.array([])
            while i not in cos_lis:
                cos_lis = np.append(cos_lis, i).astype(int)
                visited.add(i)
                i *= 2
                i %= n
            coset[leader] = cos_lis
    
    return coset

In [25]:
factors_32, _ = galois.factors(x_n)
# print(factors_32)

cyclotomic_coset = con_coset(n)
# print(cyclotomic_coset)

con_power = np.arange(1, 2*t+1)
# print(con_power)

- generator polynomial is defined as the lcm( all minimal polynomials related to the consecutive powers)

- identifying unique cosets from the consecutive powers

- cyclotomic cosets are conjugates of the base power identified using frobenius over GF(2**n)

- we find the unique minimal polynomialss and multiply them instead of taking lcm

- BCH generator polynomial can be refered as gout sometimes

- To get the minimal polynomial for a coset we use the ${"minimal\_poly"}$ method of GF32

In [32]:
g_bch = one

g_coset = np.array([])

for i in con_power:
    leader = [keys for keys, arr in cyclotomic_coset.items() if i in arr ]
    g_coset = np.append(g_coset, leader).astype(int)

g_bch_coset = np.unique(g_coset)
print(f'Cosets used to generate the generator polynomial of BCH: {g_coset}')

primitive_element = GF32.primitive_element
print(f'Primitive Element for GF({2**m}): {primitive_element}')

for i in g_bch_coset:
    g_bch *= GF32.minimal_poly(primitive_element**i)

print(f"Generator Polynomial: {g_bch}")

Cosets used to generate the generator polynomial of BCH: [1 1 3 1]
Primitive Element for GF(32): 2
Generator Polynomial: x^10 + x^9 + x^8 + x^6 + x^5 + x^3 + 1


- Using KT construction, to get unique representatives for each cyclic equivalence class, we use one of the remaining factors of GF32 and consider it as gin

- G = Gin * Gout

### Gin

- used to generate the representatives from each equivalence class

- cin (B, k_out)

- degree(Gin) = 5;  - inner cyclic code


### Gout

- degree(Gout) = 10; BCH code - outer cyclic code

- cout (k_out, n)

### G

- degree(G) = 15;

- code-word c (B, n)

In [48]:
g_bch_factors, _ = galois.factors(g_bch)
# print(g_bch_factors)

# print(factors_32[1:])

gin_choice = [i for i in factors_32[1:] if i not in g_bch_factors]
# gin = random.choice(gin_choice)
gin = gin_choice[2]
print(f"gin polynomial choosen is {gin}")

g = gin * g_bch
print(f"The polynomial g is {g}")

gin polynomial choosen is x^5 + x^4 + x^2 + x + 1
The polynomial g is x^15 + x^11 + x^10 + x^9 + x^8 + x^7 + x^5 + x^3 + x^2 + x + 1


- Construction of ${Systematic Matrix}$ for g_bch and g

- We use a different code-word basis to generate the Generator(G), G_bch, H_bch, Parity-check(H) matrices

- we generate the parity matix(P) using the basis mentioned from the paper

##### Shapes of Matrices
- G (B, n) - (16, 31)

- H (n-B, n) - (15, 31)

- G_bch (k_out, n) - (21, 31)

- H-bch (n-k_out, n) - (10, 31)

##### Note
- G @ H.T = 0

In [68]:
def con_Parity(x, n, b, g):
    S = {}

    for i in range(1, b+1):
        S[i] = x**(n-i) % g

    S = dict(sorted(S.items(), reverse=True))

    p = []
    for i, poly in S.items():
        # print(i, poly)
        row = np.zeros(n-b)
        coeffs = poly.coeffs
        row[:len(coeffs)] = coeffs[::-1]
        p.append(row)
    
    P = np.array(p)
    # print(P)
    return P

In [86]:
def con_systematic(x, n, b, g):
    
    P = con_Parity(x, n, b, g)

    g_matrix = np.hstack((-P, np.eye(b))).astype(int)
    g_matrix %= 2

    h_matrix = np.hstack((np.eye(n-b), P.T)).astype(int)

    G = galois.GF2(g_matrix)
    H = galois.GF2(h_matrix)

    return G, H

In [88]:
k_out = n - g_bch.degree
print(f'Length of BCH message: {k_out}')

k = g.degree
print(f"Degree of generator(G) polynomial: {k}")

B = n - k
print(f"Length of ACPC code (or message bits): {B}")

G, H = con_systematic(x, n, B, g)

G_bch, H_bch = con_systematic(x, n, k_out, g_bch)

print(f"Dimensions of G matrix: {G.shape}")
print(f"Dimensions of H matrix: {H.shape}")
print(f"Dimensions of G_bch matrix: {G_bch.shape}")
print(f"Dimensions of H_bch matrix: {H_bch.shape}")

Length of BCH message: 21
Degree of generator(G) polynomial: 15
Length of ACPC code (or message bits): 16
Dimensions of G matrix: (16, 31)
Dimensions of H matrix: (15, 31)
Dimensions of G_bch matrix: (21, 31)
Dimensions of H_bch matrix: (10, 31)


- Generation of look-up table for syndrome - error vectors

- To construct the CPC using KT construction we need to perform affine translation. So we choose a vector and of BCH input dimension

- Affine translation changes the code but it doesn't alter cyclical property of code 

- So using H_bch we detect the error and correct using the syndrome table(T_syn)

In [95]:
def con_Tsyndrome(H, t=2):

    T_syn = {}

    code_size = H.shape[1]

    e = galois.GF2.Zeros(code_size)
    synd = tuple((e @ H.T).tolist())
    T_syn[synd] = e

    for i in range(code_size):
        e = galois.GF2.Zeros(code_size)
        e[i] = 1
        synd = tuple((e @ H.T).tolist())
        T_syn[synd] = e

    for i, j in itertools.combinations(range(code_size), t):
        e = galois.GF2.Zeros(code_size)
        e[i], e[j] = 1, 1
        synd = tuple((e @ H.T).tolist())
        T_syn[synd] = e

    return T_syn

In [ ]:
T_syn = con_Tsyndrome(H_bch, t)

e_affine = galois.GF2.Zeros(k_out)
e_affine[0] = 1

g_affine = e_affine @ G_bch

In [130]:
msg = galois.GF2.Random(B)
print(f'Message to be transmitted: {msg}')

codeword_gen = msg @ G + g_affine
print(f"Codeword to be transmitted: {codeword_gen}")

err = galois.GF2.Zeros(n)
err[8] ^= 1
err[18] ^= 1 
v = np.roll(codeword_gen, 5) + err

s = v @ H_bch.T
print(f'Syndrome identified as {s}')

s_tuple = tuple(s.tolist())
error_vec = T_syn[s_tuple]
print(f"Corresponding error vector: {error_vec}")

v_corrected = v - error_vec
print(f"Corrected codeword: {v_corrected}")


for i in range(len(v_corrected)):
    c_hat = np.roll(v_corrected, -i) - g_affine
    # print(f"Message Recovered: {i}, {c_hat}")
    if np.all(c_hat @ H.T == 0):
        l_est = i
        msg_hat = c_hat[-B:]
        break
    
print(f"Codeshifted by {l_est} sectors")
print(f"Message Recovered: {msg_hat}")


Message to be transmitted: [1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1]
Codeword to be transmitted: [1 0 1 0 0 1 1 0 1 0 0 1 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1]
Syndrome identified as [1 1 0 0 1 0 1 0 1 1]
Corresponding error vector: [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
Corrected codeword: [0 1 1 1 1 1 0 1 0 0 1 1 0 1 0 0 1 0 0 0 1 1 1 1 1 1 1 0 0 0 0]
Codeshifted by 5 sectors
Message Recovered: [1 1 1 1 1 1 1 0 0 0 0 0 1 1 1 1]
